# Day 3 — Pandas: Data Wrangling & Time Series with NIWA Climate Data

## Objective
Load, clean, and analyze tabular climate data using Pandas, covering CSV I/O,
missing-value handling, boolean filtering, groupby aggregation, time series
resampling, and `apply` with lambda — the core data-wrangling toolkit required
before moving to GeoPandas.

## Prerequisites
- **DataFrame / Series**: Pandas' 2D table and 1D column structures
- **CSV I/O**: `pd.read_csv()`, `df.to_csv()`
- **Inspection**: `head()`, `info()`, `describe()`, `dtypes`, `shape`
- **Missing Values**: `isna()`, `dropna()`, `fillna()`, `interpolate()`
- **Filtering & Sorting**: boolean indexing, `sort_values()`, `nlargest()` / `nsmallest()`
- **groupby**: split-apply-combine aggregation, `.agg()`, `.transform()`
- **Time Series**: `pd.to_datetime()`, `resample()`, `rolling()`
- **apply + lambda**: row-wise / column-wise custom transformations

## Dataset
Simulated NIWA climate station records for 5 NZ cities (2020–2024, monthly).
Generated in Cell 2 (no external download required, fully reproducible via `np.random.seed(42)`).

| Field | Type | Description |
|---|---|---|
| `date` | str → datetime | Observation month (YYYY-MM-DD) |
| `city` | str | Auckland / Wellington / Christchurch / Hamilton / Dunedin |
| `station_id` | str | NIWA station code |
| `temp_c` | float | Monthly mean temperature (°C) |
| `rainfall_mm` | float | Monthly total rainfall (mm) |

- **Volume**: 5 cities × 12 months × 5 years = **300 records**
- **Missing values**: ~3% injected to simulate real station gaps
  (half in `temp_c`, half in `rainfall_mm`)

### Seasonal model
Both variables use the same cosine template, but with **city-specific phase and amplitude**:

```
value(month) = center + amplitude * cos(2π * (month - peak_month) / 12)
```

- **Temperature**: `peak_month = 1` (Southern Hemisphere summer) for all cities.
  ```
  temp = t_mean + t_amp * cos(2π * (month - 1) / 12) + N(0, 0.8)
  ```
- **Rainfall**: `peak_month` **varies by city**, because NZ rainfall seasonality is highly regional:
  | City | Rain peak month | Amplitude | Rationale |
  |---|---|---|---|
  | Auckland | Jul (7) | 0.35 | Winter max — rain-bearing westerlies migrate north in winter |
  | Wellington | Jul (7) | 0.30 | Same North Island westerly regime |
  | Hamilton | Jul (7) | 0.30 | Same North Island westerly regime |
  | Christchurch | Jun (6) | 0.25 | NIWA 1991–2020 normals show June wettest |
  | Dunedin | Jan (1) | 0.15 | Otago rainfall near-uniform; several stations show summer max / winter min |

  ```
  rain_factor = 1.0 + rain_amp * cos(2π * (month - rain_peak) / 12)
  rainfall    = (rain_annual / 12) * rain_factor + N(0, 15), floored at 0
  ```

- **Why multiplicative for rain, additive for temp?** Rainfall cannot be negative
  and city totals differ by 2× (618 mm vs 1240 mm), so a proportional factor keeps
  each city's seasonality scaled to its own magnitude.

⚠️ **Simplification note**: this is a *teaching* dataset. Real NZ climate is driven by
orographic lift, föhn winds, ENSO, and elevation — none of which are modelled here.
A single cosine is a baseline, not a meteorological model. Never present synthetic
data as observation.

## Tasks
- **Task 1** (Cell 3): Load CSV & inspect structure (`head`, `info`, `describe`, `dtypes`)
- **Task 2** (Cell 4): Handle missing values — detect, inspect, fill via per-city interpolation
- **Task 3** (Cell 5): Filter & sort — warmest / coldest months, verify SH seasonality
- **Task 4** (Cell 6): `groupby` aggregation — per-city, annual, and city×year summaries
- **Task 5** (Cell 7): Time series — `resample("YE")` yearly means, `rolling(12)` smoothing
- **Task 6** (Cell 8, challenge): `apply` + lambda — add temperature / rainfall category columns

## Expected results (reference)
| Checkpoint | Expected |
|---|---|
| Records | 300 rows |
| Missing after cleaning | 0 |
| Warmest months | All in Jan (Hamilton ~20.1 °C, Auckland ~20.0 °C) |
| Coldest months | All in Jul–Aug (Christchurch ~4.6 °C, Dunedin ~5.8 °C) |
| City mean temp order | Auckland 15.4 > Hamilton 14.0 > Wellington 13.1 > Christchurch 12.1 > Dunedin 11.2 |
| Temp categories | mild 140 / warm 132 / cold 26 / hot 2 |

> Note: `resample("YE")` requires pandas ≥ 2.2. On older versions use `resample("Y")`.

In [ ]:
# Generate simulated NIWA climate data (monthly, 2020-2024, 5 cities)
import numpy as np
import pandas as pd

np.random.seed(42)  # The final answer to life, the universe, and everything. ;-)

# city: (annual_mean_temp_c, temp_amp, annual_rainfall_mm, rain_peak_month, rain_amp)
city_profiles = {
    "Auckland":     (15.4, 4.0, 1210, 7, 0.35),   # Jul wettest (NIWA)
    "Wellington":   (13.2, 4.2, 1240, 7, 0.30),   # Jul wettest
    "Christchurch": (12.1, 5.5,  618, 6, 0.25),   # Jun wettest (NIWA 1991-2020)
    "Hamilton":     (14.0, 4.8, 1120, 7, 0.30),   # Jul wettest
    "Dunedin":      (11.1, 4.5,  812, 1, 0.15),   # near-uniform, slight summer max
}
station_ids = {
    "Auckland": "A64711", "Wellington": "B93451",
    "Christchurch": "C75731", "Hamilton": "D87641", "Dunedin": "E91961",
}

records = []
for year in range(2020, 2025):
    for month in range(1, 13):
        for city, (t_mean, t_amp, rain_annual, rain_peak, rain_amp) in city_profiles.items():
            # Temperature: Southern Hemisphere cycle, peak in Jan (month=1)
            seasonal = t_amp * np.cos(2 * np.pi * (month - 1) / 12)
            temp = t_mean + seasonal + np.random.normal(0, 0.8)

            # Rainfall: same cosine template, CITY-SPECIFIC peak month
            rain_factor = 1.0 + rain_amp * np.cos(2 * np.pi * (month - rain_peak) / 12)
            rainfall = (rain_annual / 12) * rain_factor + np.random.normal(0, 15)
            rainfall = max(rainfall, 0)  # no negative rainfall

            records.append({
                "date": f"{year}-{month:02d}-01",
                "city": city,
                "station_id": station_ids[city],
                "temp_c": round(temp, 1),
                "rainfall_mm": round(rainfall, 1),
            })

df_raw = pd.DataFrame(records)

# Inject ~3% missing values to simulate real station gaps
n_missing = int(len(df_raw) * 0.03)
miss_idx = np.random.choice(df_raw.index, size=n_missing, replace=False)
df_raw.loc[miss_idx[:n_missing // 2], "temp_c"] = np.nan
df_raw.loc[miss_idx[n_missing // 2:], "rainfall_mm"] = np.nan

# Save to CSV (simulates downloading from NIWA)
df_raw.to_csv("niwa_climate_2020_2024.csv", index=False)
print(f"Generated {len(df_raw)} records -> niwa_climate_2020_2024.csv")
print("Missing values injected:", df_raw[["temp_c", "rainfall_mm"]].isna().sum().to_dict())
df_raw.head()